In [ ]:
import scanpy as sc, anndata as ad, numpy as np, pandas as pd
import anndata2ri
import rpy2.rinterface_lib.callbacks
import logging
from matplotlib import pylab
import os
import sys
anndata2ri.activate()
import yaml


In [ ]:
sc.set_figure_params(dpi=100, facecolor='white', dpi_save=500)
pylab.rcParams['figure.figsize'] = (9, 9)
homeDir = os.getenv("HOME")
sys.path.insert(1, homeDir+"/utils/")
from PlotPCA_components import *
from AdataSanityCheck import *
from PurgeAdata import *
def sanitize_strings(value):
    if isinstance(value, str):
        return (
            value.replace('_', '')
            .replace('-', '')
            .replace('@', '')
            .replace(',', '')
            .replace('/', '')
            .replace(':', '')
            .replace('(', '')
            .replace(')', '')
            .replace(';', '')
            .replace(' ', '')
        )
    return value

In [ ]:
import anndata2ri
import rpy2.rinterface_lib.callbacks
import logging
anndata2ri.activate()
%load_ext rpy2.ipython
rpy2.rinterface_lib.callbacks.logger.setLevel(logging.ERROR)

# Load data

In [ ]:
import scanpy as sc
import anndata as ad
import os
import glob

# path to your dataset folder
data_dir = homeDir+"/Biermann_Atlas/data"

# find all filtered .h5 files
h5_files = sorted(glob.glob(os.path.join(data_dir, "*_filtered.h5")))
print("Found", len(h5_files), "filtered h5 files")



adatas = []
BCs = []
for f in h5_files:
    print("Loading:", os.path.basename(f))
    adata = sc.read_h5ad(f) if f.endswith(".h5ad") else sc.read_10x_h5(f)
    # assign sample ID from filename (remove extension)
    sample_id = "_".join(os.path.basename(f).replace("_filtered.h5", "").split("_")[1:])
    print(sample_id)
    adata.obs["sample"] = sample_id
    adata.uns["_source"] = f
    adata.var_names_make_unique()
    adata.obs_names_make_unique()
    adata.obs_names = [i+f"_{sample_id}" for i in adata.obs_names.tolist()]
    BCs.extend(adata.obs_names.tolist())
    adatas.append(adata)





In [ ]:
# concatenate into one AnnData
adata = ad.concat(
    adatas,
    join="inner",
)
adata.X.max()

adata.obs["sample"]

In [ ]:
MD = pd.read_csv(data_dir+"/GSE200218_sc_sn_metadata.csv.gz", index_col=0)
MD.index = [i.split("_")[0] for i in MD.index.tolist()]
MD.index = MD.index +"_"+ MD["orig.ident"]

CommonBC = list(set(MD.index.tolist()).intersection(set(adata.obs_names.tolist())))

MD = MD.loc[CommonBC]
adata = adata[CommonBC].copy()

adata.obs = pd.concat([adata.obs, MD], axis = 1)

In [ ]:
adata.obs.columns.tolist()

In [ ]:
for i in adata.obs.columns.tolist():
    if i.startswith("proportion_"):
        del adata.obs[i]
    if i.startswith("has_"):
        del adata.obs[i]
    if i.startswith("top_loss_"):
        del adata.obs[i]
    if i.startswith("top_dupli_"):
        del adata.obs[i]
adata.obs.columns.tolist()

In [ ]:
adata = adata[adata.obs["organ"] == "Brain"].copy()
pd.crosstab(adata.obs["patient"], adata.obs["ID"])

In [ ]:

adata = adata[adata.obs["cell_type_int"] != "Doublets"].copy()
adata = adata[adata.obs["cell_type_int"] != "Contamination"].copy()

mask = adata.obs[["cell_type_main","cell_type_fine","cell_type_int"]].apply(
    lambda col: col.astype(str).str.contains("Low-quality", case=False, na=False)
)


# keep only rows where no column contains "Low-quality"
adata = adata[~mask.any(axis=1)].copy()

adata

In [ ]:
adata.obs["cell_type_main"].value_counts()


In [ ]:

adata.obs["cell_type_fine"].value_counts()


In [ ]:

adata.obs["cell_type_int"].value_counts()


In [ ]:
pd.crosstab(adata.obs["cell_type_int"],adata.obs["cell_type_fine"])

In [ ]:
# We keep only SC now

adata = adata[adata.obs.sequencing == "Single nuclei"].copy()

In [ ]:
import decoupler as dc
by = "cell_type_fine"

# Let's make 1 more group by starting form the fine division and merging the categories that are too granular

In [ ]:
pd.crosstab(adata.obs["cell_type_fine"],adata.obs["cell_type_int"] )


In [ ]:
adata.obs["cell_type_curated"] = adata.obs["cell_type_fine"]
adata.obs["cell_type_curated"].unique().tolist()
adata.obs["cell_type_curated"] = adata.obs["cell_type_curated"].replace({'CD8+ T cells TOX+':'CD8+ T cells', 
'CD8+ T cells TCF7+':"CD8+ T cells",
"CAFs":"stromal",
"Activated B cells":"B cells",
"Naïve B cells":"B cells",
"Pericytes":"stromal",
"CD4+ T cells":"CD4+ T",
 "CD8+ T cells TCF7+":"CD8+ T cells",
 "CD8+ T cells TOX+":"CD8+ T cells",
 "cDC1":"DC",
 "Monocytes":"Myeloids",
 "Microglia":"Myeloids",
 "Microglia":"Myeloids",
 "cDC2":"DC",
 "Tregs":"CD4+ T",
 "MDM FTL+":"Myeloids",
 "MDM":"Myeloids",
 "Tfh-like cells":"CD4+ T",
 "DC3":"DC"})


In [ ]:
adata.obs = adata.obs.applymap(sanitize_strings)

# 2) Sanitize column names and index of MD
adata.obs.columns = adata.obs.columns.map(sanitize_strings)

In [ ]:
adata.obs 

# Save the intermediate adata

In [ ]:
colsTOretain = ['sample', 'orig.ident',
       'patient', 'ID', 'sequencing', 'organ', 'doublet_scores', 'doublet',
       'DF_score', 'barcode_all', 'cell_type_main',
       'cell_type_fine', 'cell_type_int', 'cell_cycle', 'cell_type_curated']
colsTOretain = [sanitize_strings(c) for c in colsTOretain]

for i in adata.obs.columns.tolist():
    if i not in colsTOretain:
        del adata.obs[i]
adata.write_h5ad(homeDir+"/Biermann_Atlas/adatas/Biermann_Atlas_curated.h5ad")

In [ ]:
adata

# Start DEA

In [ ]:
import decoupler as dc
by = "celltypecurated"

In [ ]:
pbulk = dc.pp.pseudobulk(
    adata=adata,
    sample_col="patient",
    groups_col=by,
    mode="sum",
)

In [ ]:
pbulk = pbulk[pbulk.obs["psbulk_cells"] > 0].copy()
pbulk.write_h5ad(homeDir+f"/Biermann_Atlas/adatas/Assembled_pbulk_by{by}.h5ad")

In [ ]:
pbulk = sc.read_h5ad(homeDir+f"/Biermann_Atlas/adatas/Assembled_pbulk_by{by}.h5ad")

In [ ]:



Counts = pbulk.to_df().T.copy()
MD = pbulk.obs.copy()
MD["cluster"] = MD[by].astype(str)
MD["sample"] = MD["patient"].astype(str)

# 1) Apply to all values in MD (every column)
MD = MD.applymap(sanitize_strings)

# 2) Sanitize column names and index of MD
MD.columns = MD.columns.map(sanitize_strings)
MD.index   = MD.index.map(sanitize_strings)

# 3) Sanitize column names and index of Counts
Counts.columns = Counts.columns.map(sanitize_strings)



MD

In [ ]:
%load_ext rpy2.ipython
pd.DataFrame.iteritems = pd.DataFrame.items

In [ ]:
%%R -i MD -i Counts -o results -o genes

library(edgeR)
library(org.Hs.eg.db)
library(AnnotationDbi)
library(stats)


cluster <- factor(MD[["cluster"]])
sample <- factor(MD[["sample"]])

y <- DGEList(Counts, group = cluster,sample=sample, genes = rownames(Counts))
keep.samples <- y$samples$lib.size > 5e4
print(table(keep.samples))



y <- y[, keep.samples]
keep.genes <- filterByExpr(y, group=y$cluster)
print(table(keep.genes))
y <- y[keep.genes, , keep=FALSE]

print("normLibSizes")
y <- normLibSizes(y)

cluster <- as.factor(y$samples$group)
donor <- factor(y$samples$sample)
design <- model.matrix(~ cluster + donor)
colnames(design) <- gsub("donor", "", colnames(design))
colnames(design)[1] <- "Int"

print(head(design))


print("estimateDisp")
y <- estimateDisp(y, design, robust=TRUE)
print("fitting")

fit <- glmQLFit(y, design, robust=TRUE)


ncls <- nlevels(cluster)
contr <- rbind( matrix(1/(1-ncls), ncls, ncls),matrix(0, ncol(design)-ncls, ncls) )
diag(contr) <- 1
contr[1,] <- 0
rownames(contr) <- colnames(design)
colnames(contr) <- paste0("cluster", levels(cluster))
contr

print(head(contr))


AllGenes <- 20000
results <- list()
for(i in colnames(contr)){
    print(sprintf("Extracting toptags for  %s",i ))
    qlf <-glmQLFTest(fit, contrast=contr[,i])
    qlf <- topTags(qlf, n=AllGenes)$table
    results[[paste0(i)]] <- qlf
}


genes <- rownames(y)

In [ ]:
import pandas as pd

resultsDict = dict(zip([str(i) for i in  list(results.names())], list(results.values())))
for k in resultsDict:
    resultsDict[k]["celltype"] = k

os.makedirs(homeDir+"/Biermann_Atlas/DEAresults", exist_ok=True)

pd.concat([resultsDict[k] for k in resultsDict.keys()], ignore_index=True).to_csv(homeDir+f"/Biermann_Atlas/DEAresults/Biermann_Atlas_DEGs_by{by}.csv")
pd.Series(list(genes)).to_csv(homeDir+f"/Biermann_Atlas/DEAresults/Biermann_Atlas_universe_by{by}.csv")